# Experiment: Case-001 Indonesia gene therapy owner verification gate

Objective: validate the July 13 cross-border evidence decision and public-safe owner-question matrix for the Indonesian pediatric thalassemia gene-therapy case report.

Success criteria: the local-access claim is deprioritized, required owner roles are present, source labels are public, and patient-action outputs remain blocked.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

ROOT = Path.cwd()
GATE_FILENAME = "2026-07-13-indonesia-gene-therapy-owner-verification-gate.json"
GATE_PATH = ROOT / "data/workflows/case-001" / GATE_FILENAME
payload = json.loads(GATE_PATH.read_text(encoding="utf-8"))
payload["current_label"]

## Plan

- Check the gate date, decision, label, and upstream Case-001 state.
- Check that each required owner role has exactly one owner-bounded question.
- Check that sources are public URLs and include the Indonesian case report plus current gene-therapy benchmarks.
- Check that allowed outputs and blocked patient-action outputs remain separated.

In [ ]:
assert payload["checked_date"] == "2026-07-13"
assert payload["workflow_id"] == "indonesia_gene_therapy_owner_verification_gate"
assert (
    payload["current_label"]
    == "indonesia_cross_border_gene_therapy_case_deprioritized_for_local_access"
)
assert payload["decision"]["state"] == "deprioritize"
assert (
    payload["upstream_state"]["case001_state"]
    == "branch_review_handoff_packet_incomplete"
)

source_labels = {source["source_label"] for source in payload["source_checks"]}
required_sources = {
    "Indonesian pediatric thalassemia gene-therapy case report",
    "TIF 2025 TDT gene manipulation chapter",
    "FDA CASGEVY product page",
    "FDA ZYNTEGLO product page",
    "ANTARA BPOM ATMP report",
    "PubMed PMID 35267028",
}
assert required_sources.issubset(source_labels)
assert all(
    source["source_url"].startswith("https://") for source in payload["source_checks"]
)
len(source_labels)

In [ ]:
required_owner_roles = {
    "treating_hematologist",
    "advanced_therapy_center_or_transplant_cell_therapy_owner",
    "regulator_or_regulatory_affairs_owner",
    "cell_manufacturing_or_gmp_owner",
    "payer_or_financial_access_owner",
    "long_term_follow_up_owner",
}
owner_roles = {question["owner_role"] for question in payload["owner_questions"]}
assert owner_roles == required_owner_roles
assert len(payload["owner_questions"]) == len(required_owner_roles)
assert all(
    question["one_question"].endswith("?") for question in payload["owner_questions"]
)
assert all(
    question["safe_state"].endswith("_needed")
    for question in payload["owner_questions"]
)
assert all(
    "Do not" in question["blocked_jump"] for question in payload["owner_questions"]
)
sorted(owner_roles)

In [ ]:
allowed = set(payload["allowed_public_outputs"])
blocked = set(payload["blocked_outputs"])
must_block = {
    "patient_specific_eligibility",
    "therapy_selection",
    "referral_instruction",
    "travel_or_import_plan",
    "purchase_or_procurement_route",
    "lab_contact_permission",
    "sample_routing",
}

assert allowed.isdisjoint(blocked)
assert must_block.issubset(blocked)
assert "case_report_not_access_proof" in allowed
assert "cross_border_case_only" in allowed
assert "local_indonesia_route_deprioritized" in allowed

summary = {
    "label": payload["current_label"],
    "decision": payload["decision"]["state"],
    "source_count": len(payload["source_checks"]),
    "owner_question_count": len(payload["owner_questions"]),
    "case001_state": payload["upstream_state"]["case001_state"],
    "patient_action_state": "blocked",
}
summary

## Results

- The primary report resolves the route as cross-border, so local Indonesian access is deprioritized.
- The remaining evidence gaps are routed to six owner categories rather than one broad public action ask.
- Each row stays source-backed, owner-bounded, and public-safe.
- Case-001 remains blocked from advanced-therapy branch movement until private packet readiness and qualified clinician review change.

## Next steps

- Verify named authorization, ethics, registry, product, and release identifiers.
- Link them to anonymized 12-month-or-longer outcome and delivered-cost evidence.
- Keep registry monitoring and any owner reply separate from patient action.